# Day4_02F_Smart_Campus_Assistant_Capstone

## Final Technical Capstone - DSU Smart Campus Assistant with OpenAI Agent + MCP

**Duration:** 45–60 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Running Example:** DSU Smart Campus Assistant  

---

## Learning Objectives

By the end of this final capstone notebook, participants will be able to:

- Build an end-to-end Agentic AI application.
- Connect an OpenAI Agent to an MCP Server.
- Use MCP tools to access campus systems.
- Understand how Agents discover and use tools.
- Extend an MCP Server without changing Agent logic.
- Design similar assistants for universities and enterprises.
- Explain the complete Agentic AI journey from LLMs to MCP.

---

## Capstone Promise

This is not just another demo.

This notebook brings together the complete learning journey:

```text
LLM
  ↓
Agent
  ↓
Tools
  ↓
Workflow
  ↓
MCP
  ↓
Enterprise AI System
```

We will build a practical DSU Smart Campus Assistant.


# 1. Opening Story

Imagine DSU wants to launch a Smart Campus Assistant.

A professor should be able to ask:

- Who is the AI&ML coordinator?
- When is the library open on Saturday?
- Tell me about AGENT401.
- What academic events are scheduled in August?
- Is the AI Lab available in the afternoon?
- When is the AI301 exam?

The assistant should not guess.

It should connect to campus systems using MCP.

That is what we will build.


# 2. Final Architecture

```text
Professor
   |
   v
DSU Smart Campus Assistant
   |
   v
OpenAI Agent SDK
   |
   v
MCP Client
   |
   v
DSU Smart Campus MCP Server
   |
   |-- faculty_directory()
   |-- library_timings()
   |-- course_information()
   |-- check_lab_availability()
   |-- academic_calendar()
   |-- exam_schedule()
```

This is a real enterprise-style architecture.

The same pattern works for:

- universities
- banks
- hospitals
- DevOps teams
- manufacturing plants
- retail platforms


# 3. What Makes This Different from a Chatbot?

A chatbot mainly answers from model knowledge.

This assistant does more.

```text
User asks question
   |
   v
Agent understands intent
   |
   v
Agent selects MCP tool
   |
   v
MCP Server accesses campus capability
   |
   v
Agent explains result
```

The Agent is not just generating text.

It is using connected capabilities.


# 4. Setup Requirements

Make sure these packages are installed:

```bash
pip install openai-agents "mcp[cli]" python-dotenv
```

Project root should contain:

```text
.env
```

with:

```text
OPENAI_API_KEY=your_key_here
```

This notebook creates the MCP server file automatically.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from agents import Agent, Runner
from agents.mcp import MCPServerStdio
from agents.model_settings import ModelSettings

# 5. Load Environment Variables

In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("OpenAI API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 6. Create the Final Smart Campus MCP Server

This server includes all the tools from the previous notebook and one new tool:

```text
exam_schedule()
```

The important learning point:

> When we add a new tool to the MCP Server, the Agent can discover it without rewriting the Agent's core logic.


In [ ]:
server_code = r"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DSU Smart Campus MCP Server - Capstone")

@mcp.tool()
def faculty_directory(department: str) -> str:
    """
    Return faculty coordinator and contact details for a department.
    Use this when the user asks about faculty, coordinator, department contact, or office details.
    """
    data = {
        "cse": "CSE Coordinator: Prof. Kumar | Room: 402 | Email: cse.office@example.edu | Extension: 4567",
        "ai&ml": "AI&ML Coordinator: Prof. Meena | Room: 305 | Email: aiml.office@example.edu | Extension: 4588",
        "aiml": "AI&ML Coordinator: Prof. Meena | Room: 305 | Email: aiml.office@example.edu | Extension: 4588",
        "ece": "ECE Coordinator: Prof. Divya | Room: 210 | Email: ece.office@example.edu | Extension: 4599",
        "mechanical": "Mechanical Coordinator: Prof. Ramesh | Room: 115 | Email: mech.office@example.edu | Extension: 4522",
    }
    return data.get(department.strip().lower(), "Department not found. Available departments: CSE, AI&ML, ECE, Mechanical.")

@mcp.tool()
def library_timings(day: str = "weekday") -> str:
    """
    Return library timings for weekdays, Saturday, or Sunday.
    Use this when the user asks whether the library is open or asks library hours.
    """
    timings = {
        "weekday": "Library Timings: Monday to Friday, 8:00 AM to 8:00 PM.",
        "saturday": "Library Timings: Saturday, 9:00 AM to 5:00 PM.",
        "sunday": "Library is closed on Sunday except during special exam periods.",
    }
    return timings.get(day.strip().lower(), "Please specify weekday, saturday, or sunday.")

@mcp.tool()
def course_information(course_code: str) -> str:
    """
    Return course details such as title, credits, semester, and outcomes.
    Use this when the user asks about a course code, course title, syllabus, credits, or outcomes.
    """
    courses = {
        "AI301": ("Artificial Intelligence", 4, "5", "Search, reasoning, knowledge representation, and AI applications."),
        "ML302": ("Machine Learning", 4, "6", "Supervised learning, unsupervised learning, model evaluation, and deployment basics."),
        "AGENT401": ("Agentic AI Systems", 3, "7", "Agents, tools, planning, multi-agent systems, workflows, and MCP integration."),
    }
    key = course_code.strip().upper()
    if key not in courses:
        return "Course not found. Available course codes: AI301, ML302, AGENT401."
    title, credits, semester, outcome = courses[key]
    return f"Course Code: {key}\\nTitle: {title}\\nCredits: {credits}\\nSemester: {semester}\\nOutcome: {outcome}"

@mcp.tool()
def check_lab_availability(lab_name: str, slot: str) -> str:
    """
    Check lab availability for a requested time slot.
    Use this when the user asks whether a lab is available.
    """
    availability = {
        ("ai lab", "morning"): "AI Lab is available in the morning slot.",
        ("ai lab", "afternoon"): "AI Lab is occupied in the afternoon. Next available slot: evening.",
        ("iot lab", "morning"): "IoT Lab is occupied in the morning. Next available slot: afternoon.",
        ("iot lab", "afternoon"): "IoT Lab is available in the afternoon slot.",
        ("lab 1", "morning"): "Lab 1 is available in the morning slot.",
        ("lab 2", "evening"): "Lab 2 is available in the evening slot.",
    }
    return availability.get((lab_name.strip().lower(), slot.strip().lower()), "Availability not found for this lab and slot.")

@mcp.tool()
def academic_calendar(month: str) -> str:
    """
    Return important academic events for a given month.
    Use this when the user asks about events, FDPs, assessments, examinations, or academic calendar.
    """
    events = {
        "july": "July Events: FDP on Agentic AI, Internal Assessment 1, Research proposal review week.",
        "august": "August Events: Mid-semester examination, Placement readiness workshop, AI project review.",
        "september": "September Events: Lab evaluation, Hackathon, Industry expert lecture series.",
    }
    return events.get(month.strip().lower(), "No events found for this month. Try July, August, or September.")

@mcp.tool()
def exam_schedule(course_code: str) -> str:
    """
    Return exam schedule for a given course code.
    Use this when the user asks about exam dates, assessment dates, or test schedule.
    """
    exams = {
        "AI301": "AI301 exam is scheduled on 18 August, 10:00 AM to 1:00 PM, Room Block A-204.",
        "ML302": "ML302 exam is scheduled on 20 August, 10:00 AM to 1:00 PM, Room Block B-105.",
        "AGENT401": "AGENT401 exam is scheduled on 22 August, 2:00 PM to 5:00 PM, AI Lab.",
    }
    key = course_code.strip().upper()
    return exams.get(key, "Exam schedule not found. Available course codes: AI301, ML302, AGENT401.")

@mcp.tool()
def register_fdp(faculty_name: str, topic: str) -> str:
    """
    Register a faculty member for an FDP topic.
    Use this when the user asks to register for an FDP or workshop.
    This is a demo tool and does not perform real registration.
    """
    return f"Demo registration completed for {faculty_name} for the FDP topic: {topic}. Please verify with the official FDP coordinator."

if __name__ == "__main__":
    mcp.run()
"""

server_path = Path("smart_campus_capstone_server.py")
server_path.write_text(server_code, encoding="utf-8")

print(f"Created final MCP server file: {server_path.resolve()}")

# 7. Configure MCP Server Parameters

We will launch the server using stdio transport.

On some systems, replace `python` with `python3`.


In [ ]:
server_params = {
    "command": "python",
    "args": ["smart_campus_capstone_server.py"],
}

server_params

# 8. Create a Reusable Campus Assistant Function

This function:

1. Starts the MCP server.
2. Creates an OpenAI Agent.
3. Attaches the MCP server.
4. Runs the user question.
5. Returns the final answer.

This keeps the notebook clean.


In [ ]:
async def ask_smart_campus_assistant(question: str):
    async with MCPServerStdio(
        name="DSU Smart Campus MCP Server - Capstone",
        params=server_params,
        cache_tools_list=True,
    ) as server:

        assistant = Agent(
            name="DSU Smart Campus Assistant",
            instructions="""
            You are the official DSU Smart Campus Assistant for faculty.

            Always use MCP tools whenever campus-specific information or actions are required.

            Use campus MCP tools for:
            - faculty or department contacts
            - library timings
            - course details
            - lab availability
            - academic calendar events
            - exam schedules
            - FDP registration

            Do not invent campus information.
            If information is not available through the MCP server, say so clearly.

            Keep responses concise, professional, and faculty-friendly.
            For action tools such as registration, remind users to verify with the official office.
            """,
            mcp_servers=[server],
            model_settings=ModelSettings(tool_choice="auto"),
        )

        result = await Runner.run(assistant, question)
        return result.final_output

# 9. Demo 1 - Faculty Directory

Question:

> Who is the AI&ML coordinator?

Expected MCP tool:

```text
faculty_directory()
```


In [ ]:
answer = await ask_smart_campus_assistant("Who is the AI&ML coordinator?")

print(answer)

# 10. Demo 2 - Library Timings

Question:

> When is the library open on Saturday?

Expected MCP tool:

```text
library_timings()
```


In [ ]:
answer = await ask_smart_campus_assistant("When is the library open on Saturday?")

print(answer)

# 11. Demo 3 - Course Information

Question:

> Tell me about AGENT401.

Expected MCP tool:

```text
course_information()
```


In [ ]:
answer = await ask_smart_campus_assistant("Tell me about AGENT401.")

print(answer)

# 12. Demo 4 - Academic Calendar

Question:

> What events are scheduled in August?

Expected MCP tool:

```text
academic_calendar()
```


In [ ]:
answer = await ask_smart_campus_assistant("What events are scheduled in August?")

print(answer)

# 13. Demo 5 - Lab Availability

Question:

> Is AI Lab available in the afternoon?

Expected MCP tool:

```text
check_lab_availability()
```


In [ ]:
answer = await ask_smart_campus_assistant("Is AI Lab available in the afternoon?")

print(answer)

# 14. Demo 6 - New Tool Discovery

This is the key MCP learning point.

We added a new tool:

```text
exam_schedule()
```

We did not change the Agent logic.

Now ask:

> When is the AI301 exam?

The Agent should discover and use the new MCP tool.


In [ ]:
answer = await ask_smart_campus_assistant("When is the AI301 exam?")

print(answer)

# 15. Demo 7 - Action Tool

Now we ask for a demo action:

> Register Prof. Suresh for Agentic AI FDP.

Expected MCP tool:

```text
register_fdp()
```

This is an action-style tool.

In production, this would require authentication and confirmation.


In [ ]:
answer = await ask_smart_campus_assistant(
    "Register Prof. Suresh for the Agentic AI FDP."
)

print(answer)

# 16. Multi-Capability Question

Now ask a question that may need multiple tools.

Example:

> Who is the AI&ML coordinator, when is the AGENT401 exam, and is the library open on Saturday?

This demonstrates orchestration through tool use.


In [ ]:
answer = await ask_smart_campus_assistant(
    "Who is the AI&ML coordinator, when is the AGENT401 exam, and is the library open on Saturday?"
)

print(answer)

# 17. What Happened Internally?

For every question, this flow happened:

```text
User Question
   |
   v
OpenAI Agent
   |
   v
Tool Selection
   |
   v
MCP Client
   |
   v
MCP Server
   |
   v
Campus Tool
   |
   v
Tool Result
   |
   v
Final Answer
```

The Agent did not need custom code for each campus system.

It used the MCP server as a standard capability layer.


# 18. Interactive Assistant Loop

The cell below gives a simple console-style assistant.

In VS Code notebooks, you can run the cell and type questions.

Type `exit` to stop.

Sample questions:

- Who is the CSE coordinator?
- Tell me about ML302.
- When is AGENT401 exam?
- Is IoT Lab available in the morning?
- Register Prof. Ravi for Agentic AI FDP.


In [ ]:
# Optional interactive demo.
# Run this only during live teaching if you want a console-style experience.

# while True:
#     question = input("Professor > ")
#     if question.lower().strip() in ["exit", "quit"]:
#         print("Assistant session ended.")
#         break
#     answer = await ask_smart_campus_assistant(question)
#     print("\nAssistant >")
#     print(answer)
#     print("-" * 80)


# 19. Tools vs Resources Reflection

In our demo, most capabilities were exposed as tools for simplicity.

In a production MCP design:

```text
Resources:
- faculty directory
- library timings
- course catalog
- academic calendar
- exam schedule

Tools:
- register FDP
- book lab
- send email
- create helpdesk ticket
```

For teaching, tools are easier to demonstrate.

For architecture, the Tool vs Resource distinction still matters.


# 20. Security and Governance

This demo is safe because it uses mock data.

In production, we need:

- Authentication
- Authorization
- Audit logging
- Human confirmation
- Role-based access
- Data masking
- Approval flows for sensitive actions

Examples:

```text
Read library timings -> low risk
Read attendance -> sensitive
Register FDP -> action
Update marks -> high risk
Cancel exam -> critical
```

Never expose powerful tools without guardrails.


# 21. Enterprise Mapping

The DSU Smart Campus Assistant pattern maps directly to enterprises.

## Banking

```text
Customer
   |
   v
Banking Assistant
   |
   v
MCP Server
   |
   |-- customer_profile
   |-- loan_status
   |-- card_blocking
   |-- complaint_ticket
```

## Healthcare

```text
Patient / Doctor
   |
   v
Hospital Assistant
   |
   v
MCP Server
   |
   |-- doctor_schedule
   |-- appointment_booking
   |-- lab_report_status
   |-- ward_availability
```

## DevOps

```text
Engineer
   |
   v
DevOps Assistant
   |
   v
MCP Server
   |
   |-- logs
   |-- metrics
   |-- deployment_status
   |-- create_incident
```


# 22. Department Assistant Design Workshop

Ask participants to design their own assistant.

Use this template:

```text
Assistant Name:

Primary Users:

Resources:
1.
2.
3.

Tools:
1.
2.
3.

Prompts:
1.
2.

Guardrails:
1.
2.
3.

Future Expansion:
1.
2.
```

Example:

```text
Assistant Name: CSE Department Assistant
Users: Faculty, students, HOD
Resources: timetable, faculty directory, course catalog
Tools: book lab, raise support request, register event
Prompts: student helpdesk prompt, placement guidance prompt
Guardrails: role-based access, approval for bookings, audit logs
```


# 23. Final Reflection Questions

Ask participants:

1. What did the LLM do?
2. What did the Agent do?
3. What did MCP do?
4. What did the tools do?
5. What should be modeled as resources?
6. Where do guardrails fit?
7. How would you productionize this?

If participants can answer these, the FDP has achieved its technical goal.


# 24. Complete AI System Mental Model

```text
LLM
  |
  v
Gives intelligence

Agent
  |
  v
Makes decisions and uses tools

Workflow
  |
  v
Coordinates multi-step processes

MCP
  |
  v
Connects AI to external systems

Guardrails
  |
  v
Keep the system safe and governed
```

Together, these form a production-oriented Agentic AI architecture.


# 25. Final FDP Evolution Map

```text
Traditional AI
   |
   v
LLMs
   |
   v
Prompt Engineering
   |
   v
Embeddings
   |
   v
RAG
   |
   v
Agents
   |
   v
Multi-Agent Systems
   |
   v
Stateful Workflows
   |
   v
MCP Integration
   |
   v
Enterprise AI Systems
```

This is the journey participants have completed.


# 26. What Participants Have Built

Across the FDP, participants have built:

- LLM prompt demos
- RAG concepts
- Agent examples
- Function tools
- Guardrails
- Multi-agent patterns
- CrewAI workflows
- LangGraph stateful workflows
- MCP server
- OpenAI Agent connected to MCP
- Smart Campus Assistant capstone

This is a complete modern Agentic AI learning path.


# 27. Common Production Considerations

Before taking such systems live, think about:

- API cost
- latency
- monitoring
- logging
- tool safety
- role-based access
- privacy
- data retention
- human approval
- fallback responses
- evaluation
- observability
- error handling

A good demo becomes a good product only with engineering discipline.


# 28. Final Instructor Closing

You can close the session with this message:

> The future is not about building smarter chatbots.
>
> The future is about building AI systems that can safely reason, collaborate, follow workflows, and connect to the real world.

That is the real promise of Agentic AI.


# 29. Capstone Exercise

Participants should create a design for one real assistant:

Choose one:

- Department Assistant
- Placement Assistant
- Research Assistant
- FDP Assistant
- Lab Assistant
- Library Assistant
- Student Mentoring Assistant

They should define:

1. User personas
2. Questions the assistant answers
3. Resources needed
4. Tools needed
5. Prompts needed
6. Guardrails needed
7. Success metrics


# 30. Key Takeaways

In this final capstone notebook, we learned:

- MCP connects Agents to real systems.
- OpenAI Agents can use MCP servers as tool providers.
- MCP servers can be extended without rewriting Agent logic.
- Tools perform actions.
- Resources provide information.
- Good metadata helps Agents choose tools correctly.
- Enterprise AI requires security, governance, and observability.
- Agentic AI is a system design discipline, not just prompting.

## Final Mental Model

```text
LLM = Intelligence
Agent = Decision maker
Crew = Collaboration
LangGraph = Workflow control
MCP = Enterprise connectivity
Guardrails = Safety
```

This completes the technical FDP content.
